In [25]:
import os
import sqlite3
import numpy as np
import pandas as pd

In [2]:
# raw_train_data = pd.read_csv("../data/raw/cs-training.csv")

# conn = sqlite3.connect("../data/raw/credit_risk.db")

# raw_train_data.to_sql(
#     "credit_risk_train",
#     conn,
#     if_exists="replace",
#     index=False
# )

### Read Dataset from SQLite DB

In [3]:
conn = sqlite3.connect("../data/raw/credit_risk.db")

raw_train_data = pd.read_sql_query(
    "SELECT * FROM credit_risk_train",
    conn
)

In [4]:
raw_train_data.shape

(150000, 12)

In [5]:
raw_train_data.head(3)

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0


In [6]:
raw_train_data.columns = raw_train_data.columns.str.lower()

In [7]:
raw_train_data.drop(["unnamed: 0"], axis=1, inplace=True)

In [8]:
raw_train_data.columns

Index(['seriousdlqin2yrs', 'revolvingutilizationofunsecuredlines', 'age',
       'numberoftime30-59dayspastduenotworse', 'debtratio', 'monthlyincome',
       'numberofopencreditlinesandloans', 'numberoftimes90dayslate',
       'numberrealestateloansorlines', 'numberoftime60-89dayspastduenotworse',
       'numberofdependents'],
      dtype='str')

In [9]:
raw_train_data["acct_id"] = "id_" + pd.Series(range(1, len(raw_train_data)+1)).astype(str)

In [10]:
raw_train_data.rename(columns = {
    "seriousdlqin2yrs" : "f_dpd_90plus_nxt_2yrs",
    "revolvingutilizationofunsecuredlines" : "avg_util_unsec",
    "numberoftime30-59dayspastduenotworse" : "n_dpd_30_50_l2yrs",
    "debtratio" : "debt_income_ratio",
    "monthlyincome" : "monthly_income",
    "numberofopencreditlinesandloans" : "n_credit_lines",
    "numberoftimes90dayslate" : "n_dpd_90plus_hist",
    "numberrealestateloansorlines" : "n_mort_loans",
    "numberoftime60-89dayspastduenotworse" : "n_dpd_60_89_l2yrs",
    "numberofdependents" : "n_dependents"
    }, inplace=True
)

In [11]:
raw_train_data.dtypes

f_dpd_90plus_nxt_2yrs      int64
avg_util_unsec           float64
age                        int64
n_dpd_30_50_l2yrs          int64
debt_income_ratio        float64
monthly_income           float64
n_credit_lines             int64
n_dpd_90plus_hist          int64
n_mort_loans               int64
n_dpd_60_89_l2yrs          int64
n_dependents             float64
acct_id                      str
dtype: object

In [12]:
raw_train_data.columns

Index(['f_dpd_90plus_nxt_2yrs', 'avg_util_unsec', 'age', 'n_dpd_30_50_l2yrs',
       'debt_income_ratio', 'monthly_income', 'n_credit_lines',
       'n_dpd_90plus_hist', 'n_mort_loans', 'n_dpd_60_89_l2yrs',
       'n_dependents', 'acct_id'],
      dtype='str')

In [13]:
raw_train_data = raw_train_data[
    ["acct_id", "n_dpd_90plus_hist", "n_dpd_60_89_l2yrs", "n_dpd_30_50_l2yrs", "n_mort_loans", "n_credit_lines",
     "n_dependents", "avg_util_unsec", "debt_income_ratio", "monthly_income", "age", "f_dpd_90plus_nxt_2yrs"]
]

In [14]:
raw_train_data.head(3)

,acct_id,n_dpd_90plus_hist,n_dpd_60_89_l2yrs,n_dpd_30_50_l2yrs,n_mort_loans,n_credit_lines,n_dependents,avg_util_unsec,debt_income_ratio,monthly_income,age,f_dpd_90plus_nxt_2yrs
0,id_1,0,0,2,6,13,2.0,0.766127,0.802982,9120.0,45,1
1,id_2,0,0,0,0,4,1.0,0.957151,0.121876,2600.0,40,0
2,id_3,1,0,1,0,2,0.0,0.658180,0.085113,3042.0,38,0


### Handling Missing Values - Median Imputation + Missing Column Indicator Method

In [15]:
raw_train_data.head(10)

,acct_id,n_dpd_90plus_hist,n_dpd_60_89_l2yrs,n_dpd_30_50_l2yrs,n_mort_loans,n_credit_lines,n_dependents,avg_util_unsec,debt_income_ratio,monthly_income,age,f_dpd_90plus_nxt_2yrs
0,id_1,0,0,2,6,13,2.0,0.766127,0.802982,9120.0,45,1
1,id_2,0,0,0,0,4,1.0,0.957151,0.121876,2600.0,40,0
2,id_3,1,0,1,0,2,0.0,0.658180,0.085113,3042.0,38,0
3,id_4,0,0,0,0,5,0.0,0.233810,0.036050,3300.0,30,0
4,id_5,0,0,1,1,7,0.0,0.907239,0.024926,63588.0,49,0
5,id_6,0,0,0,1,3,1.0,0.213179,0.375607,3500.0,74,0
6,id_7,0,0,0,3,8,0.0,0.305682,5710.000000,NaN,57,0
7,id_8,0,0,0,0,8,0.0,0.754464,0.209940,3500.0,39,0
8,id_9,0,0,0,0,2,NaN,0.116951,46.000000,NaN,27,0
9,id_10,0,0,0,4,9,2.0,0.189169,0.606291,23684.0,57,0


In [16]:
raw_train_data.isna().sum()

acct_id                      0
n_dpd_90plus_hist            0
n_dpd_60_89_l2yrs            0
n_dpd_30_50_l2yrs            0
n_mort_loans                 0
n_credit_lines               0
n_dependents              3924
avg_util_unsec               0
debt_income_ratio            0
monthly_income           29731
age                          0
f_dpd_90plus_nxt_2yrs        0
dtype: int64

In [17]:
print("missing values in following vars:", "n_dependents,", "monthly_income")
print("% missing in n_dependents:", 100 * (raw_train_data["n_dependents"].isna().sum() / raw_train_data.shape[0]))
print("% missing in monthly_income:", 100 * (raw_train_data["monthly_income"].isna().sum() / raw_train_data.shape[0]))

missing values in following vars: n_dependents, monthly_income
% missing in n_dependents: 2.616
% missing in monthly_income: 19.820666666666668


In [18]:
print("mean for n_dependents:", raw_train_data["n_dependents"].mean())
print("mean for monthly_income:", raw_train_data["monthly_income"].mean())

mean for n_dependents: 0.7572222678605657
mean for monthly_income: 6670.221237392844


In [19]:
print("median for n_dependents:", raw_train_data["n_dependents"].median())
print("median for monthly_income:", raw_train_data["monthly_income"].median())

median for n_dependents: 0.0
median for monthly_income: 5400.0


In [20]:
raw_train_data = raw_train_data.assign(
    n_dependents_nan_flag = np.where(
        raw_train_data["n_dependents"].isna(), 1, 0
    ),
    monthly_income_nan_flag = np.where(
        raw_train_data["monthly_income"].isna(), 1, 0
    )
)

In [21]:
raw_train_data["n_dependents_imp"] = raw_train_data["n_dependents"].fillna(raw_train_data["n_dependents"].median())
raw_train_data["monthly_income_imp"] = raw_train_data["monthly_income"].fillna(raw_train_data["monthly_income"].median())

In [22]:
raw_train_data.drop(columns=["n_dependents", "monthly_income"], inplace=True)

In [23]:
raw_train_data.head(10)

,acct_id,n_dpd_90plus_hist,n_dpd_60_89_l2yrs,n_dpd_30_50_l2yrs,n_mort_loans,n_credit_lines,avg_util_unsec,debt_income_ratio,age,f_dpd_90plus_nxt_2yrs,n_dependents_nan_flag,monthly_income_nan_flag,n_dependents_imp,monthly_income_imp
0,id_1,0,0,2,6,13,0.766127,0.802982,45,1,0,0,2.0,9120.0
1,id_2,0,0,0,0,4,0.957151,0.121876,40,0,0,0,1.0,2600.0
2,id_3,1,0,1,0,2,0.658180,0.085113,38,0,0,0,0.0,3042.0
3,id_4,0,0,0,0,5,0.233810,0.036050,30,0,0,0,0.0,3300.0
4,id_5,0,0,1,1,7,0.907239,0.024926,49,0,0,0,0.0,63588.0
5,id_6,0,0,0,1,3,0.213179,0.375607,74,0,0,0,1.0,3500.0
6,id_7,0,0,0,3,8,0.305682,5710.000000,57,0,0,1,0.0,5400.0
7,id_8,0,0,0,0,8,0.754464,0.209940,39,0,0,0,0.0,3500.0
8,id_9,0,0,0,0,2,0.116951,46.000000,27,0,1,1,0.0,5400.0
9,id_10,0,0,0,4,9,0.189169,0.606291,57,0,0,0,2.0,23684.0


In [26]:
file_path = "../data/processed/train_dqi_output.parquet"

if os.path.exists(file_path):
    print("File already exists!")
else:
    raw_train_data.to_parquet(
        file_path,
        index=False
    )
    print("File created successfully!")

File created successfully!
